# R0 GPU training & inference validation — 3D→2D anime renderer

**Engineer-only GPU test; not a Ghibli-style visual-quality result.** Built for temporary Google Colab runtime with public GitHub repo checkout, checkpoint persistence and research-mode teacher-target import.

Before running: **Runtime → Change runtime type → GPU**. For the first test use a T4-class accelerator if available; for 512px training prefer L4/24 GB or A100/40 GB when the account allows. 200 CCUs on Google AI Pro are a monthly balance, not guaranteed GPU hours.

No pretrained Ghibli teacher weights or model API calls are bundled. Do not put GitHub tokens into notebook source; use `SOURCE_MODE = "github"` to download from `alexliyu7352/cato` without a token. Until the repository contains the project, use `SOURCE_MODE = "upload"` with the prepared ZIP.


In [ ]:
import os, sys, json, time, shutil, zipfile, tempfile, subprocess, urllib.request, pathlib
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. Choose Runtime > Change runtime type > GPU and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(['nvidia-smi'], check=True)


## 1 — Project source

`upload`: choose the provided `Anime_Neural_Renderer_R0_Colab_Bundle_20261017.zip` (offline-safe).

`github`: download the published public `alexliyu7352/cato` `main` source archive without a token. The GitHub App still needs repository access to publish future changes; Colab only needs read access. If the repository is empty or not yet synchronized, change `SOURCE_MODE` to `upload`.


In [ ]:
SOURCE_MODE = 'github'  # use 'upload' until source files have been pushed
GITHUB_REPO = 'alexliyu7352/cato'
GITHUB_REF = 'main'
project = pathlib.Path('/content/anime_renderer_r0')
project.parent.mkdir(parents=True, exist_ok=True)
if project.exists(): shutil.rmtree(project)
with tempfile.TemporaryDirectory() as td:
    archive = pathlib.Path(td) / 'project.zip'
    if SOURCE_MODE == 'github':
        api = f'https://api.github.com/repos/{GITHUB_REPO}/zipball/{GITHUB_REF}'
        req = urllib.request.Request(api, headers={
            'Accept': 'application/vnd.github+json',
            'User-Agent': 'Cato-R0-Colab-Experiment'})
        with urllib.request.urlopen(req, timeout=75) as res, archive.open('wb') as dst:
            shutil.copyfileobj(res, dst)
    elif SOURCE_MODE == 'upload':
        from google.colab import files
        uploaded = files.upload()
        zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
        if len(zip_names) != 1:
            raise ValueError('Upload exactly one R0 Colab source ZIP archive')
        archive.write_bytes(uploaded[zip_names[0]])
    else:
        raise ValueError('SOURCE_MODE must be upload or github')
    extracted = pathlib.Path(td) / 'extracted'
    extracted.mkdir()
    with zipfile.ZipFile(archive) as z:
        bad = [m.filename for m in z.infolist() if '..' in pathlib.PurePosixPath(m.filename).parts or m.filename.startswith('/')]
        if bad: raise RuntimeError('Unsafe paths in ZIP')
        z.extractall(extracted)
    candidates = list(extracted.rglob('pyproject.toml'))
    if len(candidates) != 1: raise RuntimeError(f'Expected one project, got {len(candidates)}')
    shutil.copytree(candidates[0].parent, project)
print('Project loaded at:', project)
print('Files:', len(list(project.rglob('*.py'))))


## 2 — Install and validate the **engineering** fixture

Smoke fixtures were derived from actual 3D offscreen rendering, but target labels are algorithmic NPR, **not** any high-quality Ghibli model. It is forbidden to treat this test as evidence of artistic quality.


In [ ]:
def run(*args, cwd=None):
    print('Running:', ' '.join(str(a) for a in args))
    subprocess.run([str(a) for a in args], cwd=str(cwd or project), check=True)
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[smoke]')
fixture_zip = project / 'assets/r0_smoke_fixture.zip'
manifest = project / 'samples/smoke/manifest.jsonl'
if not manifest.is_file():
    if not fixture_zip.exists(): raise FileNotFoundError('No smoke fixture ZIP; use the Colab bundle')
    with zipfile.ZipFile(fixture_zip) as z: z.extractall(project)
assert manifest.is_file(), 'Dataset missing after fixture extraction'
run(sys.executable, '-m', 'r0.audit', '--manifest', manifest)
run(sys.executable, '-m', 'pytest', '-q')


## 3 — Short GPU training, both routes

Use tiny runs first so the first GPU experiment does not consume much of your compute quota. Results here measure the **pipeline**, not Ghibli aesthetics.


In [ ]:
smoke_output = pathlib.Path('/content/r0_runs/gpu_smoke')
SMOKE_STEPS = 50
for route in ('A','B'):
    run(sys.executable, '-m', 'r0.train', '--route', route, '--manifest', manifest,
        '--device', 'cuda', '--threads', '2', '--steps', str(SMOKE_STEPS),
        '--batch', '1', '--patch', '128', '--allow-smoke', '--out', smoke_output)
for route in ('A','B'):
    p = smoke_output / f'route_{route}_train.json'
    print(route, json.loads(p.read_text())['elapsed_sec'], 'seconds')


## 4 — Measure GPU inference, not CPU; optionally try HD

The benchmark uses `torch.cuda.synchronize` for timing and reports peak PyTorch CUDA allocation, which is **not** a whole-system VRAM measurement.


In [ ]:
gpu_bench = pathlib.Path('/content/r0_runs/gpu_benchmark')
for h,w in [(192,320), (360,640)]:
    run(sys.executable, '-m', 'r0.benchmark', '--device', 'cuda', '--precision', 'fp16',
        '--height', h, '--width', w, '--reps', '5',
        '--out', gpu_bench / f'{w}x{h}.json')
DO_HD_BENCHMARK = False
if DO_HD_BENCHMARK:
    run(sys.executable, '-m', 'r0.benchmark', '--device', 'cuda', '--precision', 'fp16',
        '--height', '720', '--width', '1280', '--reps', '3',
        '--out', gpu_bench / '1280x720.json')


## 5 — Optional real-teacher research pass (only after teacher targets are available)

Place same-size PNGs named `forest_0000.png`, `portrait_0000.png`, `village_0000.png`, etc., in a directory in Drive. At least three independent scene groups are required. **Research mode** accepts exploratory teachers without the production rights/independent-quality-approval gate; every item is marked `research_teacher_unreviewed`. No automatic artistic-quality claim.


In [ ]:
TEACHER_TARGETS = ''  # e.g. /content/drive/MyDrive/AnimeRendererR0/teacher_targets
TEACHER_MODEL_ID = 'research-style-teacher'
if TEACHER_TARGETS:
    teacher_manifest = pathlib.Path('/content/r0_runs/research_teacher/manifest.jsonl')
    run(sys.executable, '-m', 'r0.teacher_ingest', '--mode', 'research',
        '--teacher-id', TEACHER_MODEL_ID, '--original', manifest,
        '--targets', TEACHER_TARGETS, '--out', teacher_manifest)
    run(sys.executable, '-m', 'r0.audit', '--manifest', teacher_manifest)
    # In early 3-scene tests this checks gradients, not generalization. Expand first.
    for route in ('A','B'):
        run(sys.executable, '-m', 'r0.train', '--route', route, '--manifest', teacher_manifest,
            '--device', 'cuda', '--threads', '2', '--steps', '250',
            '--batch', '1', '--patch', '128', '--out', '/content/r0_runs/teacher_students')
else:
    print('No real teacher targets configured: skipping Ghibli-style training.')


## 6 — Persist artifacts **before** Colab runtime is reclaimed

Colab GPU machines are temporary. Saving to Google Drive requires authorization. The runtime does **not** push commits to GitHub; models/datasets must be archived explicitly.


In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    target = pathlib.Path('/content/drive/MyDrive/AnimeRendererR0/experiments')
    target.mkdir(parents=True, exist_ok=True)
    stamp = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
    result_zip = shutil.make_archive(f'/content/r0_gpu_results_{stamp}', 'zip', '/content/r0_runs')
    shutil.copy2(result_zip, target / pathlib.Path(result_zip).name)
    print('Saved to Google Drive:', target / pathlib.Path(result_zip).name)
else:
    print('Results remain in ephemeral /content/r0_runs; download before disconnecting.')
